# 03. heap_birthday

In [12]:
import pandas as pd
from datetime import datetime

class Heap:
    def __init__(self, *args):
        if len(args) != 0:
            self.__A = args[0]
        else:
            self.__A = []

    def insert(self, x):
        self.__A.append(x)
        self._percolateUp(len(self.__A) - 1)

    def _percolateUp(self, i):
        parent = (i - 1) // 2
        if i > 0 and self.__A[i] > self.__A[parent]:
            self.__A[i], self.__A[parent] = self.__A[parent], self.__A[i]
            self._percolateUp(parent)

    def deleteMax(self):
        if not self.isEmpty():
            max_value = self.__A[0]
            self.__A[0] = self.__A.pop()
            self._percolateDown(0)
            return max_value
        return None

    def _percolateDown(self, i):
        child = 2 * i + 1
        right = 2 * i + 2
        if child <= len(self.__A) - 1:
            if right <= len(self.__A) - 1 and self.__A[child] < self.__A[right]:
                child = right
            if self.__A[i] < self.__A[child]:
                self.__A[i], self.__A[child] = self.__A[child], self.__A[i]
                self._percolateDown(child)

    def max(self):
        return self.__A[0]

    def buildHeap(self):
        for i in range((len(self.__A) - 2) // 2, -1, -1):
            self._percolateDown(i)

    def isEmpty(self):
        return len(self.__A) == 0

    def clear(self):
        self.__A = []

    def size(self):
        return len(self.__A)

# data setting
df = pd.read_csv("DS_Birthday.csv")
df = df.dropna(subset=["생년월일8자리(예.20040101)"]).copy()
df["생년월일_문자"] = df["생년월일8자리(예.20040101)"].astype(str).str.split(".").str[0]
df = df[df["생년월일_문자"].str.len() == 8].copy()
df["생년월일"] = pd.to_datetime(df["생년월일_문자"], format="%Y%m%d", errors="coerce")
df = df[df["생년월일"].notnull()] 

birthday_heap = Heap()
for _, row in df.iterrows():
    birthday_heap.insert((row["생년월일"].timestamp(), row["이름"], row["생년월일"]))

top_10 = []
count = min(10, birthday_heap.size())   # 데이터 개수 10개 미만이면 모든 데이터 출력.

for i in range(count):
    person = birthday_heap.deleteMax()
    top_10.append(person)


print("생일이 늦은 순서 Top 10:")
for ts, name, birth in top_10:
    print(f"{name} - {birth.date()}")


생일이 늦은 순서 Top 10:
신수민 - 2005-12-30
이서영 - 2005-12-25
강민주 - 2005-12-14
김민경 - 2005-12-02
이서영 - 2005-11-12
배시은 - 2005-11-02
김여원 - 2005-10-31
이서진 - 2005-10-28
서홍빈 - 2005-10-24
김예빈 - 2005-10-19


# 04. sameteam_circulationDoublyLinkedList

In [ ]:
class BidirectNode:
    def __init__(self, item, prev=None, next=None):
        self.item = item
        self.prev = prev
        self.next = next

class CircularDoublyLinkedList:
    def __init__(self):
        self.__head = BidirectNode("dummy", None)
        self.__head.prev = self.__head
        self.__head.next = self.__head
        self.__numItems = 0

    def append(self, newItem) -> None:
        prev = self.__head.prev
        newNode = BidirectNode(newItem, prev, self.__head)
        prev.next = newNode
        self.__head.prev = newNode
        self.__numItems += 1

    def __iter__(self):
        return CircularDoublyLinkedListIterator(self)

    def getNode(self, i: int) -> BidirectNode:
        curr = self.__head
        for index in range(i + 1):
            curr = curr.next
        return curr

class CircularDoublyLinkedListIterator:
    def __init__(self, alist):
        self.__head = alist.getNode(-1)
        self.iterPosition = self.__head.next

    def __next__(self):
        if self.iterPosition == self.__head:
            raise StopIteration
        else:
            item = self.iterPosition.item
            self.iterPosition = self.iterPosition.next
            return item

    def __iter__(self):
        return self


import pandas as pd

df = pd.read_csv("DS_Birthdaydata.csv", dtype={"학번": str})

group_members = {
    "20230523", "20241207", "20241274", "20241256", "20230837",
    "20241177", "20241188", "20230875", "20241234", "20231401",
    "20222615", "20241243", "20230229"
}

cdll = CircularDoublyLinkedList()
for _, row in df.iterrows():
    item = (str(row["학번"]), row["이름"], row["생년월일8자리(예.20040101)"])
    cdll.append(item)

print("같은 조원 생일 목록:")
for item in cdll:
    student_id, name, birth = item
    if student_id in group_members:
        print(f"{name} ({student_id}) - 생일: {birth}")


# 05. 교재 8장 우선 순위 큐 연습 문제 

## 01
### 가능하다. 자식 노드의 key가 부모 노드의 key보다 작기만 하면 되므로 이것만 유지 된다면 힙의 성질에 위배 되지 않는다. 

## 02
### 항상 가장 작은 값을 가지지 않는다. 자식 노드의 key가 부모 노드의 key보다 작기만 하면 되기때문.

## 03 
### n/2. 리프노드의 개수는 n/2이다. 자식 노드가 없는 리프노드의 경우 buildHeap() 스며내리기를 할지 고려하지 않아도 되기 때문에. n/2개의 원소는 그냥 넘어가면 된다.

## 04
### 최선의 경우 Θ(1) -> 스며내리기 실행 X
### 최악의 경우 Θ(n) -> leaf노드까지 내려가는 경우. 루트노드에서부터 leaf노드까지 내려감.

## 05
### 맨 마지막 원소를 삭제하려면 루트노드부터 배열에 있는 모든 노드를 삭제해야하기 때문에 간단한 일이 아니다. / 힙에서 보다 리스트에서 맨 마지막 원소를 삭제하는 것이 더 간단함.

## 06
### 같음. 부모 노드와 값을 비교하여 크면 비교 아니면 다른 노드로 비교 교환이 넘어가는 과정은 동일하기 때문에, 위/아래 방향은 상관없이 O(n)을 동일하다.

## 07
### 특정 원소의 값이 증가하여 부모노드의 값보다 커질 수 있다. 이 경우 최대 힙의 속성이 깨지게 된다. -> 부모 노드와 값을 비교, 교환하며 루트 노드까지 이동한다. 이때 힙의 높이가 log n이므로 O(log n)시간에 성질을 복구할 수 있다. 

# 06. LeetCode 703.Kth Largest Element in Stream

In [6]:
import heapq
import ast
from typing import List

class KthLargest:

    def __init__(self, k: int, nums: List[int]):
        self.k = k
        self.min_heap = nums
        heapq.heapify(self.min_heap)

        while len(self.min_heap) > self.k:
            heapq.heappop(self.min_heap)

    def add(self, val: int) -> int:
        if (len(self.min_heap) < self.k):
            heapq.heappush(self.min_heap, val)
        elif val > self.min_heap[0]:
            heapq.heappop(self.min_heap)
            heapq.heappush(self.min_heap, val)

        return self.min_heap[0]


commands = ast.literal_eval(input())
inputs = ast.literal_eval(input())
output = []

for i in range(len(commands)):
    command = commands[i]
    arg = inputs[i]

    if (command == "KthLargest"):
        k = arg[0]
        nums = arg[1]
        outheap = KthLargest(k, nums)
        output.append(None)
    elif (command == "add"):
        output.append(outheap.add(arg[0]))
    
print(output)

[None, 4, 5, 5, 8, 8]
